In [1]:
import networkx as nx
import numpy as np
import sys
import math
import time
import random
import json
import matplotlib.pyplot as plt

In [5]:
#FUNCIONES DE GENERACIÓN DE GRAFOS E INDIVIDUOS
def graph(filename):
    c=[]
    file = open(filename, "r")
    content = file.read()
    file.close()
    
    a=content.split('\n')
    b=a[0].split(' ')
    nV=b[2]
    nE=b[3]
    a.remove(a[0])
    
    for i in a:
        c.append(i.split(' '))
    c.remove(c[-1])
    
    #Construir el grafo del problema
    G = nx.Graph()
    for i in c:
        G.add_edge(int(i[0]), int(i[1]))

    return G


def random_x(n, t=1, p=0.5):   #genera t individuos aleatorios de tamaño n
    xs=[]
    for i in range(t):
        x=[]
        for i in range(n):
            #x.append(random.randint(0,1))
            a=random.random()
            if a>p:
                x.append(1)
            else:
                x.append(0)
        xs.append(x)
    return xs

def binstrRec(s, i, res):    #función recursiva que genera en res todas las cadenas binarias 
    n = len(s)
    if i == n:
        res.append("".join(s))
        return
    s[i] = '0'
    binstrRec(s, i + 1, res)
    s[i] = '1'
    binstrRec(s, i + 1, res)
    
def binstr(n):  #funcion que devuelve todos los arrays binarios posibles de tamaño n
    s = ['0'] * n
    res = []
    binstrRec(s, 0, res)
    return res

#en una 2-coloración, los individuos serán cadenas de variables binarias en que el 1 representa un color y el 0 otro
#en el problema del clique, los individuos serán cadenas de variables binarias en que el cada 1 representa que el vértice i-ésimo pertenece al clique

def decoder(ind, G):    #decodifica individuos de cadenas de variables bianrias al subgrafo de G que formarían los nodos representados por esas variables
    F = nx.Graph()
    g_nodes=list(G.nodes())
    n=len(g_nodes)
    for i in range(len(ind)):
        if (ind[i]==1):
            F.add_node(g_nodes[i])
    f_nodes=list(F.nodes())
    for i in f_nodes:
        select_edges=list(G.edges(i))
        current_f_edges=F.edges()
        for j in select_edges:
            if not((j[0],j[1]) in current_f_edges or (j[1],j[0]) in current_f_edges) and (j[1] in f_nodes):
                F.add_edge(j[0], j[1])
    return F


def encoder(F, G):   #codifica un subgrafo de G en variables bianrias que representan si un vértice está o no incluído en el subgrafo F
   ind=[]
   g_nodes=list(G.nodes())
   f_nodes=list(F.nodes())
   for i in range(len(g_nodes)):
       val=int(g_nodes[i] in f_nodes)
       ind.append(val)
   return ind


def encoder_vertexarray(va, G):  #codifica un array con índices de vértices en un array de variables binarias que representan si un vértice de G está o no incluído en va
    ind=[]
    g_nodes=list(G.nodes())
    for i in range(len(g_nodes)):
        val=int(g_nodes[i] in va)
        ind.append(val)
    return ind
    

In [3]:
#Para guardar una intancia de colorig clique (en formato grafo), se usa nx.write_adjlist(CC, filename)
#Para cargar una intancia de colorig clique (en formato grafo), se usa CC=nx.read_adjlist(filename)

#Para guardar y cargar los archivos de resultados se usan las funciones siguientes:
def save_instance_file(sat_instance_descriptor, file_path="instance.json"):  
    json_str = json.dumps(sat_instance_descriptor)
    file_path 
    with open(file_path, 'w') as file:
        json.dump(json_str, file)
    return 0

def load_file_instance(file_path='instance.json'):
    file = open(file_path,)
    data = json.load(file)
    array = json.loads(data)
    return array

In [3]:
def check_clique(F):   #Dado un subgrafo F de G, esta función comprueba si es un clique
    f_nodes= F.nodes()
    nf= len(f_nodes)
    clique_val=(len(list(F.edges()))>0)
    if clique_val:
        for i in f_nodes:
            if not(len(F.edges(i)) == (nf-1)):
                clique_val=False
                break;
    return clique_val
            

def check_maximalclique(F, G):    #si F es un clique de G, entonces esta función comprueba si el clique es maximal
    f_nodes= list(F.nodes())
    g_edges= list(G.edges())
    maxclique_val=True
    conected_nodes=[]
    
    for i in f_nodes:            #se recogen todos los nodos conectados en G a los nodos de F 
        inode_g_edges=list(G.edges(i))
        for j in inode_g_edges:
            if ((not (j[1] in conected_nodes)) and (not (j[1] in f_nodes))):
                conected_nodes.append(j[1])

    conected_sum=0
    total=len(f_nodes)
    for i in conected_nodes:    #se comprueba si alguno de los nodos conectados en G a los nodos de F está conectado en G a todos los nodos de F    
        conectednode_g_edges= list(G.edges(i))
        conected_sum=0
        for j in conectednode_g_edges:
            if (j[1] in f_nodes):
                conected_sum+=1
        if total==conected_sum:
            maxclique_val=False
            break;
            
    return maxclique_val           


In [5]:
def check_coloring(coloring_ind, cliques, G):   #con un ind de primer nivel (2-coloración del grafo G en formato grafo) y un array de cliques de G (en formato de subgrafos) se calcula si todos los los cliques contienen los dos colores. La función devuelve [validez_global, validez de cada clique]  
    #K=decoder(coloring_ind, G) 
    #k_nodes= list(K.nodes())
    k_nodes=list(coloring_ind.nodes())
    bi_colored_cliques=[]
    val_cliques=[1]
    for cliq in cliques:
        cl_nodes=list(cliq.nodes())
        nodos_color_k=0
        for cl_n in cl_nodes:
            if cl_n in k_nodes:
                nodos_color_k+=1
        if ((nodos_color_k==len(cl_nodes)) or (nodos_color_k==0)):
            bi_colored_cliques.append(0)
            val_cliques[0]=0
        else:
            bi_colored_cliques.append(1)
    val_cliques.append(bi_colored_cliques)
    return val_cliques

In [7]:
#GENERACCIÓN DE INSTANCIAS
def generate_instance(n, density=0.1, edge_prob=0.7):    #recibe el tamaño del grafo, una medida de densidad de aristas (como referencia, un 0.1 indica que cada nodo puede tener hasta n/100 aristas y un 1 indica que cada nodo puede tener hasta n/10 aristas) y una probabilidad de llenarlas (entre 0 y1)
    max_edges=round((n*0.1)*density)
    G = nx.Graph()
    for i in range(n):
        G.add_node(i+1)
    free_nodes=list(G.nodes())
    for i in range(n):  #(i+1 es el nodo de G objetivo de la iteración)
        edge_chances=int(max_edges-len(list(G.edges(i+1))))   #a cada nodo se le dan tantas oportunidades de añadir una nueva arista como huecos para aristas tenga el nodo
        if(edge_chances>0):
            for j in range(edge_chances):    
                chance=random.random()
                if (chance<=edge_prob):
                    free_nodes_posible_conections=free_nodes.copy()
                    free_nodes_posible_conections.remove(i+1)
                    node_new_edge=random.choice(free_nodes_posible_conections)    #de los nodos que queden con huecos libres para aristas, se escoge uno aleatorio
                    G.add_edge(i+1, node_new_edge)
                    if(len(list(G.edges(node_new_edge)))>=max_edges):   #si el nodo destino de la arista ha alcanzado el máximo con esta, se le elimina de la lista de nodos con huecos libres para aristas
                       free_nodes.remove(node_new_edge)
            if(len(list(G.edges(i+1)))>=max_edges):    #si todas las oportunidades han añadido una arista, se elimina el nodo de la iteración de la lista de nodos con huecos libres para aristas
                free_nodes.remove(i+1)
        elif (i+1 in free_nodes):     #si ya estaba al máximo pero aún estaba en la lista, se elimina el nodo de la arista (si lo anterior funciona bien no debería entrar nunca aquí)
            free_nodes.remove(i+1)

    for node in free_nodes:
        if len(list(G.edges(node)))==0:
            free_nodes_posible_conections=free_nodes.copy()
            free_nodes_posible_conections.remove(node)
            node_new_edge=random.choice(free_nodes_posible_conections) 
            G.add_edge(node, node_new_edge)
    
    return G


In [9]:
#RESOLUBILIDAD DE INSTANCIAS
def bruteforce_cliquecoloring(G):     #función para comprobar por fuerza brusta si existe alguna 2-coloración del grafo G que cubra con 2 colores distintos los vértices cada uno de todos los cliques maximal de G
    g_nodes=list(G.nodes())
    ind_len=len(g_nodes)
    posible_colorations_string=binstr(ind_len)   #se obtienen todas las posibles 2-coloraciones de G
    posible_colorations_string.pop(0)
    posible_colorations_string.pop(-1)
    posible_colorations=[]
    posible_clique_maximal=list(nx.find_cliques(G))   #se obtienen todos los cliques maximal de G
    nodes_colorations=[]
    total_colorations_check=[]
    total_bi_colorations=[]
    val=0

    #y=[]
    #for string in posible_colorations_string:
    #    for i in range(len(string)):
    #        y.append(int(string[i]))
    #    posible_colorations.append(y)
    #    y=[]

    for coloration_string in posible_colorations_string:     #para cada posible coloración
        coloration=[]
        for i in range(len(coloration_string)):
            coloration.append(int(coloration_string[i]))
        coloration_bi_colorated_cliques=[]
        for clique in posible_clique_maximal:    #se comprueba cada clique maximal de G
            nodes_col_clique=0
            for i in range(ind_len):
                if coloration[i] and (str(i+1) in clique):   #se cuenta cada nodo del clique que tenga uno de los colores 
                    nodes_col_clique+=1
            if nodes_col_clique<len(clique) and nodes_col_clique>0:   #y se comprueba que no todos tengan el mismo color
                coloration_bi_colorated_cliques.append(clique)            #de ser así se añaden a una lista de los cliques bi-coloreados por esta coloraciónç
            cliques_bi_colorated_sum=len(coloration_bi_colorated_cliques)
        total_colorations_check.append([coloration, cliques_bi_colorated_sum, coloration_bi_colorated_cliques])   #y se lleva registro que asocia una coloración con los cliques que bi-colorea 
        if (cliques_bi_colorated_sum==len(posible_clique_maximal)):     #si todos los cliques maximal de G están en la lista de cliques bi-coloreados por la coloración
            total_bi_colorations.append(coloration)    #se añade la coloración a una lista de soluciones válidas de la instancia
            val=1     #y se marca el problema como resoluble

    bi_colorations_sum=len(total_bi_colorations)
    mono_colorations_sum=len(posible_colorations_string)-bi_colorations_sum
    bi_cloloration_ratio= bi_colorations_sum/mono_colorations_sum

    #la función devuelve si la instancia del problema es resoluble, la proporción de coloraciones que la resuleven y las coloraciones que lo hacen (el conjunto de soluciones validas)
    return val, bi_cloloration_ratio, total_bi_colorations


def check_coloring_true_val(coloration, G, posible_clique_maximal_input=None, verbose=False):    #función para comprobar por fuerza bruta si una de las coloraciones es una solución válida del problema
    g_nodes=list(G.nodes())
    ind_len=len(g_nodes)
    if(posible_clique_maximal_input==None):
        posible_clique_maximal=list(nx.find_cliques(G)) 
    else:
        posible_clique_maximal=posible_clique_maximal_input
    total_valid_cliques=0
    for clique in posible_clique_maximal:
        nodes_col_clique=0
        for i in range(ind_len):
            if (coloration[i]==1) and (str(i+1) in clique):   #se cuenta cada nodo del clique que tenga uno de los colores 
                nodes_col_clique+=1
        if nodes_col_clique<len(clique) and nodes_col_clique>0:   #y se comprueba que no todos tengan el mismo color
            total_valid_cliques+=1
        if(verbose):
            print(clique, nodes_col_clique)
    if total_valid_cliques==len(posible_clique_maximal):  
        val=True
    else:
        val=False
    return val, total_valid_cliques
    

def bruteforce_cliquecoloring_check_val(G): #función para resolución completa por fuerza bruta, pero usando la función check_coloring_true_val de comprobaciones parciales
    g_nodes=list(G.nodes())
    ind_len=len(g_nodes)
    posible_colorations_string=binstr(ind_len)   #se obtienen todas las posibles 2-coloraciones de G
    posible_colorations_string.pop(0)
    posible_colorations_string.pop(-1)
    posible_clique_maximal=list(nx.find_cliques(G)) 
    #posible_colorations=[]
    total_bi_colorations=[]
    val=0

    for coloration_string in posible_colorations_string:     #para cada posible coloración
        coloration=[]
        for i in range(len(coloration_string)):
            coloration.append(int(coloration_string[i]))
        valid_coloration=check_coloring_true_val(coloration, G, posible_clique_maximal) #se comprueba si es válida
        if(valid_coloration[0]):
            total_bi_colorations.append(coloration)    #y si lo es se añade la coloración a una lista de soluciones válidas de la instancia y se marca como resoluble
            val=1
    
    bi_colorations_sum=len(total_bi_colorations)
    bi_colorations_ratio=bi_colorations_sum/len(posible_colorations_string)

    return val, bi_colorations_ratio, bi_colorations_sum 
    #return val, bi_colorations_ratio, total_bi_colorations   
    

In [11]:
#FUNCIÓN DE CRUCE
def cruce_1P(population, prob_mut=0.01):   #función de cruce en un punto
    indices_pob=[]
    hijos=[]
   
    tamanho=len(population[0])
     
    for i in range(len(population)):
        indices_pob.append(i)
        
    for i in range(int((len(population))/2)):
        
        p1=random.choice(indices_pob)
        padre1= population[p1]
        indices_pob.remove(p1)
        p2=random.choice(indices_pob)
        padre2=population[p2]
        indices_pob.remove(p2)
        
        punto_corte=random.choice(list(range(tamanho-1))[1:])
        padre1_1=padre1[:punto_corte]
        padre1_2=padre1[punto_corte:]
        padre2_1=padre2[:punto_corte]
        padre2_2=padre2[punto_corte:]
        hijo1=padre1_1 + padre2_2
        hijo2=padre2_1 + padre1_2
        
        for i in range(len(hijo1)):
            muta1=random.random()
            muta2=random.random()
            if muta1<=prob_mut:
                hijo1[i]=int(not hijo1[i])
            if muta2<=prob_mut:
                hijo2[i]=int(not hijo2[i])
        hijos.append(hijo1)
        hijos.append(hijo2)
        
        #print(padre1, padre2, punto_corte)
        #print(hijo1, hijo2)
    
    return hijos

In [9]:
#Funciones de fitness y selección

#calcular fitness mediante check_clique(clique), check_maximalclique(clique, G) y check_coloring(coloring_ind, cliques, G) para comprobar si es clique, si es maximal y si la coloración lo clubre
#los posibles fintess son [0] si no es un clique, [1,0] si es un clique no maximal, [1,1,0] si es un clique maximal no cubierto por la coloración y [1,1,1] si es un clique maximal cubierto por la coloración 
#la prioridad en el segundo nivel debería ser [1,1,0]>[1,1,1]>[1,0]>[0] para priorizar los cliques maximal no cubiertos por la coloración y los cliques maximal sobre el resto
def fitness(x, clique, G):
    fit_checkClique=check_clique(clique)
    if(fit_checkClique):
        fit=[1]
        fit_checkCliqueMaximal=check_maximalclique(clique, G)
        if(fit_checkCliqueMaximal):
            fit.append(1)
            ver_coloration=check_coloring(x, [clique], G)[0]
            fit.append(ver_coloration)
        else:
            fit.append(0)
    else:
        fit=[0]
    return fit

#función de selección del segundo nivel
def selection_elitism_lower_cc(x, pop, hijos, fit_list, G):   
    new_gen=[]
    new_fit_list=[]
    cliques_maximal=[]
    cliques_not_maximal=[]
    not_cliques=[]
    tam_pop=len(pop)
    
    for hijo in hijos:    #para cada hijo se calcula su fitness
        clique_hijo=decoder(hijo, G)
        fit=fitness(x, clique_hijo, G)
        pop.append(hijo)
        fit_list.append(fit)
        
    for i in range(len(pop)):    #se dividen los individuos en función del grado de clique que representan 
        pop[i]=([pop[i], fit_list[i]])
    for ind in pop:
        match len(ind[1]):
            case 3:
                cliques_maximal.append(ind)
            case 2:
                cliques_not_maximal.append(ind)
            case 1:
                not_cliques.append(ind)
            case _: 
                not_cliques.append(ind)

    #se ordenan los individuos por su fitness ([1,1,0]>[1,1,1]>[1,0]>[0]), se priorizan los cliques maximal que la coloración no cubra
    if(len(cliques_maximal)>0):
        cliques_maximal=list(sorted(cliques_maximal, key=lambda fit: fit[1][2])) 
    pop_sorted=cliques_maximal+cliques_not_maximal+not_cliques
    
    for i in range(tam_pop):   #los individuos con el fitness más favorable sobreviven
       new_gen.append(pop_sorted[i][0])
       new_fit_list.append(pop_sorted[i][1])
        
    return new_gen, new_fit_list


#función de selección del primer nivel
def selection_elitism_upper_cc(pop, hijos, fit_list, CC, swarmsize_lower=40, iterations_lower=50, prob_mutacion=0.01, muestreo=1):   
    new_gen=[]
    new_fit_list=[]
    cliques_maximal=[]
    cliques_not_maximal=[]
    not_cliques=[]
    tam_pop=len(pop)
    
    for hijo in hijos:    #para cada hijo se calcula su fitness
        coloration=decoder(hijo, CC)
        fit=Genetic_lower_coloring_clique(n, coloration, CC, swarmsize_lower, iterations_lower, prob_mutacion, muestreo)
        pop.append(hijo)
        fit_list.append(fit[1])
        
    for i in range(len(pop)):    #se dividen los individuos en función del grado de clique que representan 
        pop[i]=([pop[i], fit_list[i]])
    for ind in pop:
        match len(ind[1]):
            case 3:
                cliques_maximal.append(ind)
            case 2:
                cliques_not_maximal.append(ind)
            case 1:
                not_cliques.append(ind)
            case _: 
                not_cliques.append(ind)

    #se ordenan los individuos por su fitness ([1,1,1]>[1,1,0]>[1,0]>[0]), se priorizan los cliques maximal que la coloración sí cubra
    if(len(cliques_maximal)>0):
        cliques_maximal=list(reversed(sorted(cliques_maximal, key=lambda fit: fit[1][2])))
    pop_sorted=cliques_maximal+cliques_not_maximal+not_cliques
    
    for i in range(tam_pop):   #los individuos con el fitness más favorable sobreviven
       new_gen.append(pop_sorted[i][0])
       new_fit_list.append(pop_sorted[i][1])
        
    return new_gen, new_fit_list

In [21]:
#ALGORITMOS GENÉTICOS
def Genetic_lower_coloring_clique(n, x, G, swarmsize=40, iterations=50, prob_mutacion=0.01, muestreo=1):  #Algoritmo genético de segundo nivel
    #G=nx.read_adjlist(filename)
    pop=random_x(n, swarmsize)

    fit_list=[]
    for ind in pop:
        clique=decoder(ind, G)
        fit=fitness(x, clique, G)
        fit_list.append(fit)

    new_gen=pop.copy()
    new_fit_list=fit_list.copy()

    for i in range(iterations):
        hijos=cruce_1P(new_gen, prob_mutacion)
        new_gen, new_fit_list= selection_elitism_lower_cc(x, new_gen, hijos, new_fit_list, G)

    best_ind=new_gen[0]
    best_fit=new_fit_list[0]
    
    if(muestreo and len(best_fit)==3):    #se realiza el muestreo del fitness
        random_ind_index=random.sample(range(3, swarmsize-1), 3)
        ind_random_muestreo_1=new_fit_list[random_ind_index[0]]
        ind_random_muestreo_2=new_fit_list[random_ind_index[1]]
        ind_random_muestreo_3=new_fit_list[random_ind_index[2]]
        inds_muestreo=[new_fit_list[1], new_fit_list[2], ind_random_muestreo_1, ind_random_muestreo_2, ind_random_muestreo_3]
        for ind in inds_muestreo:
            if(len(ind)==3):
                best_fit[2]+=ind[2]
        best_fit[2]=best_fit[2]/6

    return best_ind, best_fit


def Genetic_coloring_clique(n, CC, swarmsize=40, iterations=50, swarmsize_lower=40, iterations_lower=50, prob_mutacion=0.01, muestreo=1):  #Algoritmo genético de primer nivel
    #CC=nx.read_adjlist(filename)
    pop=random_x(n, swarmsize)

    fit_list=[]
    for ind in pop:
        coloration=decoder(ind, CC)
        fit=Genetic_lower_coloring_clique(n, coloration, CC, swarmsize_lower, iterations_lower, prob_mutacion, muestreo)
        fit_list.append(fit[1])

    new_gen=pop.copy()
    new_fit_list=fit_list.copy()

    for i in range(iterations):
        hijos=cruce_1P(new_gen, prob_mutacion)
        new_gen, new_fit_list= selection_elitism_upper_cc(new_gen, hijos, new_fit_list, CC, swarmsize_lower, iterations_lower, prob_mutacion, muestreo=1)
        
    best_ind=new_gen[0]
    best_fit=new_fit_list[0]

    return best_ind, best_fit

In [13]:
#Función para pruebas de estabilidad del AG binivel sobre instancias de coloring-clique
def pruebas_coloring_clique_ag(n, file_names, partial_results_filename, full_solutions_filename=None, n_pruebas=10, par_comb=[[[40, 50], [40, 50]]], verbose=True):
    resultados_totales=[]
    for h in par_comb:     #para cada par de tamaños de iteraciones del ag inferior
        
        resultados_totales.append([[h[1][0], h[1][1]]])
        if(full_solutions_filename):
            cc_instance_full_solutions=load_file_instance(full_solutions_filename)
        if(verbose):
            print("Pop: ", h[1][0], "- Iteraciones: ", h[1][1])
        t1_global=time.time()
                
        for i in range(len(file_names)):   #se ejecutan n_pruebas sobre cada una de las 60 instanicas
            resultados_instancia=[]
            
            file_name=file_names[i]       #carga de la instancia y su solución completa
            CC=nx.read_adjlist(file_name)

            if(verbose and full_solutions_filename):
                print(i, file_name," - ", cc_instance_full_solutions[i][2])
            elif(verbose):
                 print(i, file_name)
                
            resultados_instancia.append(i)
            resultados_instancia.append(file_name)
            tiempo_total=0
            resultados_validos_totales=0
            
            for j in range(n_pruebas):     #prueba unitaria para una instancia (se ejecuta n_pruebas veces)
                valid_solution=0
                
                t1=time.time()
                result_ag=Genetic_coloring_clique(n, CC, swarmsize=h[0][0], iterations=h[0][1], swarmsize_lower=h[1][0], iterations_lower=h[1][1])
                t2=time.time()
                t=t2-t1
                
                tiempo_total+=t
                validity=check_coloring_true_val(result_ag[0], CC) #comprobación de validez de la solución del ag
                if(validity[0]):    
                    valid_solution=1
                resultados_validos_totales+=valid_solution
                
                resultados_instancia.append([j, result_ag[0], result_ag[1], valid_solution, t])

                if(verbose):
                    print("- ", resultados_instancia[-1])
    
            porcentaje_resultados_validos=resultados_validos_totales/n_pruebas      #estadísticas del conjunto de pruebas de la instncia
            tiempo_medio=tiempo_total/n_pruebas
            
            resultados_instancia.append(porcentaje_resultados_validos)
            resultados_instancia.append(tiempo_medio)
            resultados_instancia.append(tiempo_total)
    
            t2_global=time.time()
            t_global=t2_global-t1_global
    
            resultados_totales[-1].append(resultados_instancia)
            
            if(verbose):       #resumen de las ejecuciones de la instancia
                print(i+1, "/", len(file_names), " --- ", porcentaje_resultados_validos, " --- ", tiempo_medio, " --- ", t_global, '\n')
    
        save_instance_file(resultados_totales, partial_results_filename)

    return resultados_totales


In [ ]:
#Función para generación de lotes de instancias de coloring-clique
def full_batch_instance_generation(n=20, density=[5, 8, 10], file_generic_name="ColoringClique_instance",  num_instances_gen_objective=60, num_inst_per_dens=20, verbose=True):
    file_generic=file_generic_name+".json"
    resultados_check=[]
    resultados=[]
    i=0
    num_valid_instances=0

    t1_global=time.time()

    dens_index=0
    j=num_inst_per_dens
    #while(num_valid_instances<num_instances_gen_objective):
    for i in range(num_instances_gen_objective):
        cc_instance_descriptor=generate_instance(n,density[dens_index])   #se genera la instancia de coloring-clique
        nx.write_adjlist(cc_instance_descriptor, file_generic)
        cc_instance_descriptor=nx.read_adjlist(file_generic)
        
        t1=time.time()
        brute_force_check_full=bruteforce_cliquecoloring_check_val(cc_instance_descriptor)   #se comprueba la validez y el numero de soluciones de la instancia
        t2=time.time()
        t=t2-t1
        resultados_check.append([i, brute_force_check_full[0], brute_force_check_full[1], brute_force_check_full[2], t])
        if(verbose):
            print(resultados_check[-1])
                  
        #if(resultados_check[-1][1]==1):   #sea la instancia resoluble o no
        file_name=file_generic_name+"_"+str(i)+".json"
        nx.write_adjlist(cc_instance_descriptor, file_name)     #se guarda la instancia
        resultados.append([file_name, resultados_check[-1][1], resultados_check[-1][2], density[dens_index], t])  #se guardan el número y proporción de sus soluciones validas
        if(resultados_check[-1][1]==1):
            num_valid_instances+=1
        if(verbose):
            print(resultados[-1])
        full_sol_name=file_generic_name+"_data_solutions.json"
        save_instance_file(resultados, full_sol_name)    #y estos datos en un fichero
        

        j-=1
        if(j==0):
            dens_index+=1
            j=num_inst_per_dens

        t2_global=time.time()
        t_global=t2_global-t1_global
        if(verbose):
            print(i, "/", num_instances_gen_objective, " --- ", num_valid_instances, " --- ", t_global, '\n')
        #i+=1
    
    return resultados

In [13]:
results_CCn15_generation=load_file_instance("ColoringClique_instance_n15_data_solutions.json")
file_names_n15=[]
for instance in results_CCn15_generation:
    file_names_n15.append(instance[0])
file_names_n15

['ColoringClique_instance_n15_0.json',
 'ColoringClique_instance_n15_1.json',
 'ColoringClique_instance_n15_2.json',
 'ColoringClique_instance_n15_3.json',
 'ColoringClique_instance_n15_4.json',
 'ColoringClique_instance_n15_5.json',
 'ColoringClique_instance_n15_6.json',
 'ColoringClique_instance_n15_7.json',
 'ColoringClique_instance_n15_8.json',
 'ColoringClique_instance_n15_9.json',
 'ColoringClique_instance_n15_10.json',
 'ColoringClique_instance_n15_11.json',
 'ColoringClique_instance_n15_12.json',
 'ColoringClique_instance_n15_13.json',
 'ColoringClique_instance_n15_14.json',
 'ColoringClique_instance_n15_15.json',
 'ColoringClique_instance_n15_16.json',
 'ColoringClique_instance_n15_17.json',
 'ColoringClique_instance_n15_18.json',
 'ColoringClique_instance_n15_19.json',
 'ColoringClique_instance_n15_20.json',
 'ColoringClique_instance_n15_21.json',
 'ColoringClique_instance_n15_22.json',
 'ColoringClique_instance_n15_23.json',
 'ColoringClique_instance_n15_24.json',
 'Coloring

In [22]:
n=15
n_pruebas=5
file_names=file_names_n15
full_solutions_filename="ColoringClique_instance_n15_data_solutions.json"
partial_results_filename="ColoringClique_n15_bilevel_genetic_algorithm_experiments_results_partial.json"
par_comb=[[[20, 50], [20, 50]], [[20, 50], [20, 25]], [[20, 50], [20, 100]]]

resultados_experimentos_ag_n15=pruebas_coloring_clique_ag(n, file_names, partial_results_filename, full_solutions_filename, n_pruebas, par_comb, verbose=True)

Pop:  20 - Iteraciones:  50
0 ColoringClique_instance_n15_0.json  -  0.3580540804492462
-  [0, [0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0], [1, 1, 1.0], 0, 114.87227535247803]
-  [1, [0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0], [1, 1, 1.0], 0, 117.18560719490051]
-  [2, [1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1.0], 1, 118.92626929283142]
-  [3, [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1], [1, 1, 1.0], 0, 117.81413197517395]
-  [4, [0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1], [1, 1, 1.0], 0, 117.40279388427734]
1 / 60  ---  0.2  ---  117.24021553993225  ---  586.2030782699585 

1 ColoringClique_instance_n15_1.json  -  0.27644509552584995
-  [0, [1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0], [1, 1, 1.0], 0, 115.00460934638977]
-  [1, [1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1], [1, 1, 1.0], 0, 113.93738794326782]
-  [2, [0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1], [1, 1, 1.0], 1, 114.19697761535645]
-  [3, [0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1], [1, 1, 1.

KeyboardInterrupt: 

In [23]:
n=15
n_pruebas=5
file_names=file_names_n15
full_solutions_filename="ColoringClique_instance_n15_data_solutions.json"
partial_results_filename="ColoringClique_n15_bilevel_genetic_algorithm_experiments_results_partial.json"
par_comb=[[[20, 50], [20, 50]], [[20, 50], [20, 25]], [[20, 50], [20, 100]]]

resultados_experimentos_ag_n15=pruebas_coloring_clique_ag(n, file_names, partial_results_filename, full_solutions_filename, n_pruebas, par_comb, verbose=True)

Pop:  20 - Iteraciones:  50
0 ColoringClique_instance_n15_0.json  -  0.004394799487273393
-  [0, [1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0], [1, 1, 1.0], 0, 84.0127112865448]
-  [1, [0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1], [1, 1, 1.0], 0, 82.73018765449524]
-  [2, [0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0], [1, 1, 1.0], 0, 82.6603569984436]
-  [3, [0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1], [1, 1, 1.0], 0, 82.46877717971802]
-  [4, [0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1], [1, 1, 1.0], 0, 82.78430986404419]
1 / 60  ---  0.0  ---  82.93126859664918  ---  414.6649377346039 

1 ColoringClique_instance_n15_1.json  -  0.013428553988890923
-  [0, [1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1], [1, 1, 1.0], 0, 84.41127729415894]
-  [1, [1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0], [1, 1, 1.0], 0, 84.38987231254578]
-  [2, [0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0], [1, 1, 1.0], 0, 84.37534260749817]
-  [3, [0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0], [1, 1, 1.0], 0, 8

In [24]:
save_instance_file(resultados_experimentos_ag_n15, "ColoringClique_n15_bilevel_genetic_algorithm_experiments_results.json")

0

In [25]:
results_CCn20_generation=load_file_instance("ColoringClique_instance_n20_data_solutions.json")
file_names_n20=[]
for instance in results_CCn20_generation:
    file_names_n20.append(instance[0])
file_names_n20

['ColoringClique_instance_n20_0.json',
 'ColoringClique_instance_n20_1.json',
 'ColoringClique_instance_n20_2.json',
 'ColoringClique_instance_n20_3.json',
 'ColoringClique_instance_n20_4.json',
 'ColoringClique_instance_n20_5.json',
 'ColoringClique_instance_n20_6.json',
 'ColoringClique_instance_n20_7.json',
 'ColoringClique_instance_n20_8.json',
 'ColoringClique_instance_n20_9.json',
 'ColoringClique_instance_n20_10.json',
 'ColoringClique_instance_n20_11.json',
 'ColoringClique_instance_n20_12.json',
 'ColoringClique_instance_n20_13.json',
 'ColoringClique_instance_n20_14.json',
 'ColoringClique_instance_n20_15.json',
 'ColoringClique_instance_n20_16.json',
 'ColoringClique_instance_n20_17.json',
 'ColoringClique_instance_n20_18.json',
 'ColoringClique_instance_n20_19.json',
 'ColoringClique_instance_n20_20.json',
 'ColoringClique_instance_n20_21.json',
 'ColoringClique_instance_n20_22.json',
 'ColoringClique_instance_n20_23.json',
 'ColoringClique_instance_n20_24.json',
 'Coloring

In [ ]:
n=20
n_pruebas=3
file_names=file_names_n20
full_solutions_filename="ColoringClique_instance_n20_data_solutions.json"
partial_results_filename="ColoringClique_n20_bilevel_genetic_algorithm_experiments_results_partial.json"
par_comb=[[[20, 50], [20, 50]], [[20, 50], [20, 25]], [[20, 50], [20, 100]]]

resultados_experimentos_ag_n20=pruebas_coloring_clique_ag(n, file_names, partial_results_filename, full_solutions_filename, n_pruebas, par_comb, verbose=True)

Pop:  20 - Iteraciones:  50
0 ColoringClique_instance_n20_0.json  -  0.0010509511012098335


In [ ]:
save_instance_file(resultados_experimentos_ag_n20, "ColoringClique_n20_bilevel_genetic_algorithm_experiments_results.json")

In [ ]:
results_CCn25_generation=load_file_instance("ColoringClique_instance_n25_data_solutions.json")
file_names_n25=[]
for instance in results_CCn25_generation:
    file_names_n25.append(instance[0])
file_names_n25

In [ ]:
n=25
n_pruebas=5
file_names=file_names_n25
#full_solutions_filename="ColoringClique_instance_n25_data_solutions.json"
partial_results_filename="ColoringClique_n25_bilevel_genetic_algorithm_experiments_results_partial.json"
par_comb=[[[20, 50], [20, 50]], [[20, 50], [20, 25]], [[20, 50], [20, 100]]]

resultados_experimentos_ag_n25=pruebas_coloring_clique_ag(n, file_names, partial_results_filename, n_pruebas=n_pruebas, par_comb=par_comb, verbose=True)

In [ ]:
save_instance_file(resultados_experimentos_ag_n25, "ColoringClique_n25_bilevel_genetic_algorithm_experiments_results.json")